In [2]:
import gurobipy as gp
import numpy as np
import pandas as pd
from gurobipy import GRB


In [ ]:
import rasterio
import numpy as np

tif_path = "ForUSTree_2018_HighVeg_TreeCoverage.tif"

with rasterio.open(tif_path) as src:
    A = src.read(1)          # first band as a 2D numpy array
    nodata = src.nodata
    transform = src.transform
    crs = src.crs

print("Shape (rows, cols):", A.shape)
print("dtype:", A.dtype)
print("nodata:", nodata)
print("CRS:", crs)

n, p = A.shape

Shape (rows, cols): (1376, 1542)
dtype: uint8
nodata: 0.0
CRS: EPSG:6344


: 

Defining Constants

In [ ]:
#Placeholder for material cost of planting 1 tree
t = 10

#Placeholder for site cost of planting 1 tree

siteCost = 20

#Placeholder for max # of trees at 1 location
maxT = 100

#Placeholder for budget

budget = 1000000



#Data

#Placeholder for Impervious
a = np.random.randint(0, 2, size=(n, p))


In [ ]:
model = gp.Model("MILP")


#Creating vectors / matrices to make the constraints easier
ones_n = np.ones(n)
ones_p = np.ones(p)
U = np.ones(shape=(n, k))  *maxT

x = model.addMVar(shape=(n, p), vtype= GRB.CONTINUOUS, name = "x")

y = model.addMVar(shape = (n, p), vtype = GRB.CONTINOUS, name="y")



#Constraint to ensure we don't go over budget
model.addConstr((ones_n @ x @ ones_p)*t + (ones_n @ y @ ones_p)*siteCost <= budget)

#Constraint to ensure we don't plant more than the max at 1 site
model.addConstr(x - y*U <= 0)



#Ensuring that we only plant trees in areas we are allowd to
model.addConstr(y - a <= 0)



In [6]:
m = gp.Model("t")

lam = 90000

x_1 = m.addVar(vtype=GRB.INTEGER, lb=0)
x_2 = m.addVar(vtype=GRB.INTEGER, lb=0)

m.addConstr(-x_1 + x_2 <= 1)
m.setObjective(x_1 + x_2 + lam*(-1 - x_1 + x_2), GRB.MAXIMIZE)

m.optimize()

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 1 rows, 2 columns and 2 nonzeros
Model fingerprint: 0x4845120c
Variable types: 0 continuous, 2 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [9e+04, 9e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 1.0000000
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 14 available processors)

Solution count 1: 1 
No other solutions better than 0

Model is unbounded
Best objective 1.000000000000e+00, best bound -, gap -
